In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
# config = CNNExperiments()
config.debug = False
config.repeats = 1
config.gpu_id = 1

exp = ExperimentRun(config=config)

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
model = config.models[0]
optimizer = config.optimisers[-1]
batch = 10
gpu_id = 1
task_id = None
in_docker =False

In [5]:
exp.prepare_monte_carlo_experiments_data(number=10)

In [5]:
exp.add_task(
    model_name=model,
    batch_size=batch,
    optimizer=optimizer,
    gpu_id=0,
    zero_out=0
)
exp.add_task(
    model_name=model,
    batch_size=batch,
    optimizer=optimizer,
    gpu_id=0,
    zero_out=1
)


## Measure Ground Truth and Estimated Memory for Each job

In [6]:
exp.run_group_truth(in_docker=in_docker)

  0%|          | 0/2 [00:00<?, ?it/s]/home/glaswegian/miniconda3/envs/xmem-2.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2/2 [00:25<00:00, 12.92s/it]


## Estimate Max GPU Memory by DNNmem

In [7]:
est_list = [
    SummarySectionName.DNNmem,
    SummarySectionName.schedtune
]
if not isinstance(exp._config, CNNExperiments):
    est_list.append(SummarySectionName.LLmem)
exp.run_estimation(
    estimators=est_list,
    in_docker=in_docker
)

if isinstance(exp._config, CNNExperiments):
    exp.verify_llmem_result()



================== Create docker containers ==================


100%|██████████| 10/10 [00:20<00:00,  2.06s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 10/10 [03:52<00:00, 23.26s/it]

================== Statistics ==================
Run(success/total): 30/30


## Estimate Max GPU Memory by SchedTune

In [8]:
exp.statistics()
results = exp.to_evaluation_result()

100%|██████████| 11/11 [00:00<00:00, 11141.59it/s]


=============== Statistics for Transformer-Exp ==================
train: 11/11
config: 11/11
groundtruth: 11/11
solution: 10/11
schedtune: 11/11
DNNmem: 11/11
LLmem: 0/11


100%|██████████| 11/11 [00:00<00:00, 5215.03it/s]
